In [1]:
# Install or update the Ultralytics YOLO library to the latest version
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.9 MB/s eta 0:00:00


In [2]:
# Import Google Drive module from Google Colab
from google.colab import drive
# Mount Google Drive to access files and datasets inside Colab
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Create datasets directory if it does not exist
!mkdir -p /content/datasets

# Extract the parking dataset into the datasets folder
!unzip -q "/content/drive/MyDrive/Colab Notebooks/parking.v1i.yolov5pytorch.zip" -d /content/datasets

# List extracted dataset files
!ls /content/datasets

data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [4]:
# Display files inside the training folder
!ls /content/datasets/train

images	labels


In [5]:
# Import PyTorch library
import torch

# Save the original torch.load function
old_load = torch.load

# Create a modified load function
def fixed_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return old_load(*args, **kwargs)

# Replace torch.load with the modified version
torch.load = fixed_load

In [6]:
# Import YOLO model from Ultralytics
from ultralytics import YOLO

# Load the pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
# Train the YOLOv8 model on the parking dataset

results = model.train(
    data='/content/datasets/data.yaml',
    epochs=15,
    imgsz=640,
    batch=8,
    name='parking_yolov8'
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=parking_yolov8, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

In [9]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2342.9±888.0 MB/s, size: 70.1 KB)
val: Scanning /content/datasets/valid/labels.cache... 2483 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2483/2483 867.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 156/156 3.7it/s 42.0s
                   all       2483     143316      0.997      0.997      0.994      0.951
           space-empty       2062      73629      0.998      0.996      0.995      0.962
        space-occupied       1967      69687      0.996      0.999      0.994      0.941
Speed: 1.4ms preprocess, 3.8ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/val-2
mAP50: 0.9943903997971385
mAP50-95: 0.9512612414038664
Precision: 0.997133264790302
Recall: 0.9971894912922228


In [10]:
# Load best trained model
model = YOLO('/content/runs/detect/parking_yolov8/weights/best.pt')

In [11]:
# Run detection on test images
results = model.predict(
    source='/content/datasets/test/images',
    conf=0.25,
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1242 /content/datasets/test/images/2012-09-11_15_53_00_jpg.rf.e21d9980b1b6ccc44fbe4cc51219e879.jpg: 640x640 29 space-emptys, 72 space-occupieds, 7.3ms
image 2/1242 /content/datasets/test/images/2012-09-11_16_48_36_jpg.rf.e3b1b1ba91af0a3707e1d1c9b6bada40.jpg: 640x640 25 space-emptys, 75 space-occupieds, 7.2ms
image 3/1242 /content/datasets/test/images/2012-09-12_06_36_36_jpg.rf.0091ce85bd55d841f2ebe4f83ddf18b7.jpg: 640x640 100 space-emptys, 7.2ms


In [12]:
!cp /content/runs/detect/parking_yolov8/weights/best.pt /content/drive/MyDrive/